In [0]:
%python
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%python
df = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
df.display()

In [0]:
%python
df.createOrReplaceTempView("csv_view")


In [0]:
%python
display(spark.sql("select * from csv_view"))

In [0]:
from pyspark.sql import Row

data = [
    (1, "John", "US", 1200.50, "2026-01-15"),
    (2, "Alice", "UK", 850.75, "2026-02-10"),
    (3, "David", "FR", 1500.00, "2026-03-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df1 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df1")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS learnspark.raw.orders
          (
              customer_id INT,
              customer_name STRING,
              country STRING,
              sales_amount DOUBLE,
              order_date STRING
          )
          """)

In [0]:
spark.sql("""
          INSERT INTO learnspark.raw.orders
          SELECT * FROM df1
          """)

In [0]:
spark.sql("select * from learnspark.raw.orders").display()

In [0]:
spark.sql("""DELETE FROM learnspark.raw.orders where customer_id in (2) and sales_amount = '960.75'""")

In [0]:
from pyspark.sql import Row

data = [
    (4, "Ajay", "DE", 1590.00, "2026-12-15"),
    (2, "Alice", "UK", 960.75, "2026-02-10"),
    (5, "David", "IT", 1250.00, "2026-10-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df2 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df2")

### MergeInto Statement

In [0]:
(df2.alias("src").mergeInto("learnspark.raw.orders",col("src.customer_id") == col("learnspark.raw.orders.customer_id"))
 .whenMatched().updateAll()
 .whenNotMatched().insertAll()
 .merge()
)
 

In [0]:
%sql
EXPLAIN select * from learnspark.raw.orders

In [0]:
%sql
MERGE INTO target_table AS target
USING source_table AS source
ON target.id = source.id

WHEN MATCHED THEN
    UPDATE SET
        target.name = source.name,
        target.salary = source.salary

WHEN NOT MATCHED THEN
    INSERT (id, name, salary)
    VALUES (source.id, source.name, source.salary);